# PHẦN 1: TIỀN XỬ LÝ VÀ DỌN DẸP DỮ LIỆU - CYBERSECURITY INTRUSION DETECTION

## 1. Định nghĩa vấn đề

### Mô tả
- phát hiện và phân loại các cuộc tấn công mạng (intrusion detection) dựa trên tập dữ liệu lưu lượng mạng và hành vi người dùng.  
- dữ liệu ban đầu bao gồm 1 file: cybersecurity_intrusion_data.csv.  
- link tập dữ liệu: https://www.kaggle.com/datasets/dnkumars/cybersecurity-intrusion-detection-dataset  

### Mục tiêu
- xem thông tin chung của tập dữ liệu.  
- kiểm tra tính toàn vẹn của dữ liệu.  
- xử lý các giá trị thiếu (missing values).  

## 2. Chuẩn bị vấn đề

### 2.1. Import các thư viện cần thiết

In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)

### 2.2. Đọc dữ liệu từ file csv

In [3]:
df = pd.read_csv("../data/cybersecurity_intrusion_data.csv", low_memory=False, skipinitialspace=True)
df.head()

,session_id,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
0,SID_00001,599,TCP,4,492.983263,DES,0.606818,1,Edge,0,1
1,SID_00002,472,TCP,3,1557.996461,DES,0.301569,0,Firefox,0,0
2,SID_00003,629,TCP,3,75.044262,DES,0.739164,2,Chrome,0,1
3,SID_00004,804,UDP,4,601.248835,DES,0.123267,0,Unknown,0,1
4,SID_00005,453,TCP,5,532.540888,AES,0.054874,1,Firefox,0,0


 ### 2.3. Liệt kê các thông tin cơ bản của dữ liệu
 

In [5]:
print("\n=== THÔNG TIN TỔNG QUAN VỀ DỮ LIỆU (info) ===")
df.info()

print("\n=== THỐNG KÊ MÔ TẢ (describe) ===")
print(df.describe(include='all'))  # include='all' để xem cả biến số và biến phân loại

# Kiểm tra phân phối của biến mục tiêu 'attack_detected'
print("\n=== PHÂN PHỐI NHÃN (attack_detected) ===")
print(df["attack_detected"].value_counts())
print(f"Tỷ lệ tấn công: {df['attack_detected'].mean():.2%}")



=== THÔNG TIN TỔNG QUAN VỀ DỮ LIỆU (info) ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9537 entries, 0 to 9536
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   session_id           9537 non-null   object 
 1   network_packet_size  9537 non-null   int64  
 2   protocol_type        9537 non-null   object 
 3   login_attempts       9537 non-null   int64  
 4   session_duration     9537 non-null   float64
 5   encryption_used      7571 non-null   object 
 6   ip_reputation_score  9537 non-null   float64
 7   failed_logins        9537 non-null   int64  
 8   browser_type         9537 non-null   object 
 9   unusual_time_access  9537 non-null   int64  
 10  attack_detected      9537 non-null   int64  
dtypes: float64(2), int64(5), object(4)
memory usage: 819.7+ KB

=== THỐNG KÊ MÔ TẢ (describe) ===
       session_id  network_packet_size protocol_type  login_attempts  \
count        9537        

 ### 2.4. Xử lý dữ liệu không toàn vẹn (thiếu giá trị)

In [6]:
# Kiểm tra các cột có giá trị null
print("\n=== THỐNG KÊ GIÁ TRỊ THIẾU (MISSING VALUES) TRƯỚC KHI XỬ LÝ ===")
missing_before = df.isnull().sum()
print(missing_before[missing_before > 0])

# Xác định các dòng bị thiếu
null_positions = {
    col: df.index[df[col].isna()].tolist()
    for col in df.columns
    if df[col].isna().any()
}

print("\n=== CHI TIẾT CÁC DÒNG BỊ THIẾU DỮ LIỆU ===")
for col, idx_list in null_positions.items():
    print(f"{col}: {idx_list}")

# Nhận xét: Dữ liệu cybersecurity thường có rất ít hoặc không có missing values.
# Nếu có, do số lượng ít, có thể bỏ qua các dòng đó.
# Lưu ý: Nên in ra vài dòng bị thiếu để kiểm tra nội dung trước khi quyết định xóa.

if any(df.isnull().any()):
    print("\nHiển thị một vài dòng bị thiếu để kiểm tra:")
    rows_with_null = df[df.isnull().any(axis=1)]
    print(rows_with_null.head())
    
    # Quyết định: Xóa các dòng có missing value (vì số lượng thường rất ít)
    df = df.dropna()
    print(f"\nĐã xóa {len(rows_with_null)} dòng có giá trị thiếu.")
else:
    print("\nDữ liệu không có giá trị thiếu nào. Rất tốt!")

# Kiểm tra lại sau khi xử lý
print("\n=== KIỂM TRA LẠI KHÔNG CÒN GIÁ TRỊ THIẾU ===")
print(df.isnull().sum().sum() == 0)


=== THỐNG KÊ GIÁ TRỊ THIẾU (MISSING VALUES) TRƯỚC KHI XỬ LÝ ===
encryption_used    1966
dtype: int64

=== CHI TIẾT CÁC DÒNG BỊ THIẾU DỮ LIỆU ===
encryption_used: [8, 9, 12, 14, 17, 21, 26, 32, 34, 36, 40, 43, 50, 53, 64, 65, 66, 74, 75, 79, 83, 85, 106, 107, 117, 119, 122, 127, 132, 148, 150, 151, 158, 161, 164, 167, 171, 181, 184, 185, 195, 202, 212, 216, 222, 223, 231, 238, 241, 242, 249, 251, 255, 269, 273, 274, 275, 282, 285, 295, 299, 313, 317, 328, 335, 339, 340, 343, 346, 347, 350, 354, 358, 364, 375, 380, 388, 393, 395, 397, 401, 407, 412, 414, 418, 426, 428, 441, 444, 452, 456, 457, 467, 468, 470, 479, 491, 499, 502, 507, 512, 523, 524, 525, 530, 536, 540, 547, 549, 557, 569, 570, 591, 592, 607, 616, 617, 631, 633, 636, 638, 640, 645, 650, 651, 653, 657, 659, 664, 671, 676, 677, 684, 685, 692, 695, 704, 707, 714, 723, 724, 726, 739, 746, 748, 754, 755, 767, 772, 787, 791, 795, 800, 802, 804, 810, 812, 816, 819, 820, 822, 826, 828, 830, 831, 832, 837, 838, 852, 854, 865, 866, 

### 2.5. Xử lý các cột không đúng kiểu dữ liệu

In [7]:
# Kiểm tra kiểu dữ liệu hiện tại
print("\n=== KIỂU DỮ LIỆU CÁC CỘT ===")
print(df.dtypes)

# Xác định các cột số nhưng đang ở dạng object (do có ký tự đặc biệt hoặc khoảng trắng)
# Đối với file này, các cột số đã đúng kiểu (int64, float64).
# Tuy nhiên, một số cột phân loại (protocol_type, encryption_used, browser_type) nên giữ nguyên là object để sau này one-hot encoding.

# Kiểm tra cột 'session_id': có thể bỏ qua vì là mã định danh
if 'session_id' in df.columns:
    print("\nCột 'session_id' là định danh duy nhất, có thể bỏ qua để giảm chiều dữ liệu.")
    # df = df.drop(columns=['session_id'])  # Có thể drop nếu muốn, nhưng tạm thời giữ lại để minh họa

# Kiểm tra cột 'unusual_time_access': kiểu int64 là phù hợp
print("\nGiá trị duy nhất của cột 'unusual_time_access':", df['unusual_time_access'].unique())


=== KIỂU DỮ LIỆU CÁC CỘT ===
session_id              object
network_packet_size      int64
protocol_type           object
login_attempts           int64
session_duration       float64
encryption_used         object
ip_reputation_score    float64
failed_logins            int64
browser_type            object
unusual_time_access      int64
attack_detected          int64
dtype: object

Cột 'session_id' là định danh duy nhất, có thể bỏ qua để giảm chiều dữ liệu.

Giá trị duy nhất của cột 'unusual_time_access': [0 1]


 ### 2.6. Xóa các cột có quá nhiều giá trị trùng lặp hoặc không cần thiết

In [8]:
# Kiểm tra số lượng giá trị duy nhất của từng cột
print("\n=== SỐ LƯỢNG GIÁ TRỊ DUY NHẤT CỦA CÁC CỘT ===")
unique_counts = df.nunique()
print(unique_counts)

# Cột 'session_id' có giá trị duy nhất cho mỗi hàng → không mang lại thông tin cho mô hình.
# Nên xóa cột này để tránh overfitting.
if 'session_id' in df.columns:
    df = df.drop(columns=['session_id'])
    print("\nĐã xóa cột 'session_id'.")

# Các cột khác như 'protocol_type', 'encryption_used', 'browser_type' có số lượng giá trị duy nhất nhỏ, giữ lại để encoding.


=== SỐ LƯỢNG GIÁ TRỊ DUY NHẤT CỦA CÁC CỘT ===
session_id             7571
network_packet_size     941
protocol_type             3
login_attempts           13
session_duration       7566
encryption_used           2
ip_reputation_score    7571
failed_logins             6
browser_type              5
unusual_time_access       2
attack_detected           2
dtype: int64

Đã xóa cột 'session_id'.


 ### 2.7. Xóa các cột có tất cả các giá trị đều giống nhau (constant columns)


In [9]:
# Tìm các cột chỉ có 1 giá trị duy nhất
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
print(f"\nCác cột chỉ có 1 giá trị duy nhất (có thể xóa): {constant_cols}")

if constant_cols:
    df = df.drop(columns=constant_cols)
    print(f"Đã xóa {len(constant_cols)} cột không mang thông tin.")
else:
    print("Không có cột nào bị hằng số.")


Các cột chỉ có 1 giá trị duy nhất (có thể xóa): []
Không có cột nào bị hằng số.


### 2.8. Kiểm tra dữ liệu trước khi xuất ra file


In [10]:
# Thông tin tổng quan sau khi xử lý
print("\n=== DỮ LIỆU SAU KHI TIỀN XỬ LÝ ===")
print(f"Kích thước: {df.shape[0]} hàng, {df.shape[1]} cột")
print("Các cột còn lại:", list(df.columns))

# Kiểm tra lại giá trị thiếu
print("\nTổng số giá trị thiếu sau xử lý:", df.isnull().sum().sum())

# Xem thử vài dòng sau xử lý
print("\n5 dòng dữ liệu sau xử lý:")
print(df.head())

# Thống kê nhanh biến mục tiêu
print("\nPhân phối nhãn 'attack_detected' sau xử lý:")
print(df['attack_detected'].value_counts())


=== DỮ LIỆU SAU KHI TIỀN XỬ LÝ ===
Kích thước: 7571 hàng, 10 cột
Các cột còn lại: ['network_packet_size', 'protocol_type', 'login_attempts', 'session_duration', 'encryption_used', 'ip_reputation_score', 'failed_logins', 'browser_type', 'unusual_time_access', 'attack_detected']

Tổng số giá trị thiếu sau xử lý: 0

5 dòng dữ liệu sau xử lý:
   network_packet_size protocol_type  login_attempts  session_duration  \
0                  599           TCP               4        492.983263   
1                  472           TCP               3       1557.996461   
2                  629           TCP               3         75.044262   
3                  804           UDP               4        601.248835   
4                  453           TCP               5        532.540888   

  encryption_used  ip_reputation_score  failed_logins browser_type  \
0             DES             0.606818              1         Edge   
1             DES             0.301569              0      Firefox   
2  

### 2.9. Lưu tập dữ liệu đã làm sạch dưới dạng file parquet (hoặc csv)


In [11]:
df.to_parquet("../data_processed/cleaned_data.parquet")